In [ ]:
import pandas as pd
from discovery_utils.utils.llm import batch_check

import pandas as pd
import os

# Alternative: s3://discovery-iss/data/afs_scanning/afs_open_alex_scan_parenting_interventions.csv
INPUT_DATA = 'afs_open_alex_scan_parenting_interventions.csv'

data_df = (
    pd.read_csv(INPUT_DATA)
    .query("publication_year >= 2000")
    .assign(text = lambda df: df['title'] + ' ' + df['abstract'])
)
len(data_df)

# testing on a smaller sample for now
data_df = data_df.sample(100, random_state=42)

/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_87842/3065306713.py:10: DtypeWarning: Columns (52) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(INPUT_DATA)


In [3]:
from ast import literal_eval

def extract_author_countries(authorships):
    try:
        # Convert the string representation of the list to an actual list
        authors = literal_eval(authorships)
        countries = []
        for author in authors:
            for institution in author.get('institutions', []):
                if 'country_code' in institution:
                    countries.append(institution['country_code'])
        # remove null values
        countries = [country for country in countries if country is not None]
        return sorted(list(set(countries)))
    except (ValueError, SyntaxError):
        # Handle the case where the string cannot be evaluated
        return []

In [4]:
_data_df = data_df

In [5]:
ids = _data_df.id.tolist()
text = data_df.text.tolist()
test_data = dict(zip(ids, text))

In [6]:
len(test_data)

100

In [7]:
system_message = """
    Extract structured information about a parenting programme from the provided text.
    We are looking for programmes that are evidence-based (i.e. rigorously evaluated) and targeted at parents from pregnancy up to when their
    child is 5 years old.

    The programme could be relevant to developmental, behavioural, or well-being outcomes
    such as children language development, cognitive development, social-emotional skills, physical and mental health, or
    parental outcomes such as confidence, skills, behaviours, knowledge, self-efficacy, and well-being.
    
    Unless requested otherwise, adhere as precisely as possible to the language and text that is used in the provided text document.
    
    If the requested information is not described, return N/A. DO NOT make up any false information or false inferences.
"""

fields = [
    {"name": "is_relevant", "type": "str", "description": "One-word answer: 'yes' if the text is about a parenting programme targeted at parents at any stage from pregnancy up to when the child is 5 years old, otherwise 'no'."},
    {"name": "is_relevant_reason", "type": "str", "description": "Short explanation (one sentence, 20 words) of why the text is relevant or not."},
    
    {"name": "programme_name", "type": "str", "description": "The name of the programme or intervention."},
    {"name": "summary", "type": "str", "description": "A brief summary of what the programme involves and what it aims to achieve."},
    
    {"name": "programme_evaluation", "type": "str", "description": "'yes' if the programme has been rigorously evaluated, otherwise 'no'."},
    {"name": "programme_evaluation_desc", "type": "str", "description": "Description of how the programme was evaluated."},
    
    {"name": "digital_component", "type": "str", "description": "One-word answer: 'yes' if programme includes a digital component (e.g., text messages, online modules), otherwise 'no'."},
    {"name": "digital_component_description", "type": "str", "description": "Brief description of the digital component, if any."},
    
    {"name": "country", "type": "str", "description": "The country or countries where the programme was delivered."},
    
    {"name": "age_range", "type": "str", "description": "The programme is aimed at parents of what age group? E.g., 'pregnancy','0-5 years', 'infants', 'toddlers'."},
    {"name": "age_range_numerical", "type": "str", "description": "Infer the probable age range in years and in numerical format (e.g., '0-5', '0-2', '3-5'). This should be '-1' if the programme is aimed at pregnancy."},
    
    {"name": "child_outcomes_targeted", "type": "list[str]", "description": "Child-related outcomes the programme seeks to improve (e.g., 'language', 'cognitive development', 'social-emotional skills')."},
    {"name": "parent_outcomes_targeted", "type": "list[str]", "description": "Parent-related outcomes the programme targets (e.g., 'confidence', 'knowledge', 'wellbeing', 'skills')."},
    
    {"name": "target_population", "type": "str", "description": "Description of the parent population the programme is aimed at (e.g., low-income mothers, ethnic minorities, parents under 25)."},
    
    {"name": "disadvantaged_groups_engaged", "type": "list[str]", "description": "List of any specific disadvantaged groups the programme has successfully engaged, if described."},
    
    {"name": "engagement_strategies", "type": "list[str]", "description": "Recruitment and engagement strategies used, if described."},
    {"name": "engagement_success", "type": "str", "description": "Any information about how successful the engagement strategies were."},
    {"name": "barriers_to_engagement", "type": "list[str]", "description": "Barriers or challenges mentioned regarding recruitment or engagement."},
    
    {"name": "evidence_of_effectiveness", "type": "list[str]", "description": "Evidence of outcomes, effectiveness, or evaluations cited."},
    {"name": "limitations", "type": "list[str]", "description": "Any limitations of the programme or study mentioned."},
    
    {"name": "other_learnings", "type": "list[str]", "description": "Other useful insights about programme delivery, recruitment, or engagement, if any."}
]

In [ ]:
processor = batch_check.LLMProcessor(
    model_name="gpt-4.1-mini",
    temperature=0,
    output_path="afs_open_alex_scan_1.jsonl",
    system_message=system_message,
    session_name="afs_open_alex_scan",
    output_fields=fields,
)

processor.run(test_data, batch_size=10, sleep_time=0.5)

2025-05-15 16:40:47,123 - root - INFO - Using OpenAI
2025-05-15 16:40:47,339 - langfuse - WARNING - Langfuse client is disabled since no public_key was provided as a parameter or environment variable 'LANGFUSE_PUBLIC_KEY'. See our docs: https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client


<Task pending name='Task-5' coro=<LLMProcessor.process_text_data() running at /Users/rosie.oxbury/Documents/git_repos/discovery_utils/discovery_utils/utils/llm/batch_check.py:120>>

2025-05-15 16:40:47,352 - root - INFO - Processing batch 1/10
2025-05-15 16:40:58,020 - root - INFO - Processing batch 2/10
2025-05-15 16:41:06,957 - root - INFO - Processing batch 3/10
2025-05-15 16:41:14,840 - root - INFO - Processing batch 4/10
2025-05-15 16:41:26,735 - root - INFO - Processing batch 5/10
2025-05-15 16:41:34,192 - root - INFO - Processing batch 6/10
2025-05-15 16:41:41,099 - root - INFO - Processing batch 7/10
2025-05-15 16:41:48,840 - root - INFO - Processing batch 8/10
2025-05-15 16:41:57,211 - root - INFO - Processing batch 9/10
2025-05-15 16:42:05,679 - root - INFO - Processing batch 10/10


In [9]:
output_df = pd.read_json("afs_open_alex_scan_1.jsonl", lines=True)
output_df

,is_relevant,is_relevant_reason,programme_name,summary,programme_evaluation,programme_evaluation_desc,digital_component,digital_component_description,country,age_range,...,engagement_strategies,engagement_success,barriers_to_engagement,evidence_of_effectiveness,limitations,other_learnings,id,timestamp,model,temperature
0,no,The text discusses school climate and disorder...,N/A,N/A,no,N/A,no,N/A,N/A,N/A,...,[],N/A,[],[],[],[],https://openalex.org/W2163583898,2025-05-15 15:40:47.354145+00:00,gpt-4.1-mini,0
1,no,The text describes parental experiences with c...,N/A,N/A,no,N/A,no,N/A,Australia,N/A,...,[],N/A,[],[],[],[Parents need better support structures to cop...,https://openalex.org/W2160350695,2025-05-15 15:40:47.359249+00:00,gpt-4.1-mini,0
2,no,The text discusses trauma-informed care in mat...,N/A,N/A,no,N/A,no,N/A,N/A,N/A,...,[],N/A,[],[],[],[],https://openalex.org/W2771031857,2025-05-15 15:40:47.359888+00:00,gpt-4.1-mini,0
3,no,The text is a meta-analysis on postdivorce chi...,N/A,N/A,no,N/A,no,N/A,N/A,N/A,...,[],N/A,[],[],[],[],https://openalex.org/W2074352831,2025-05-15 15:40:47.360462+00:00,gpt-4.1-mini,0
4,no,"The text discusses genetics and social class, ...",N/A,N/A,no,N/A,no,N/A,N/A,N/A,...,[],N/A,[],[],[],[],https://openalex.org/W2150034613,2025-05-15 15:40:47.360953+00:00,gpt-4.1-mini,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,no,The text discusses skills development in the i...,N/A,N/A,no,N/A,no,N/A,"Ghana, Kenya, Nigeria, Rwanda, Tanzania",N/A,...,[],N/A,[],[],[],[The informal sector is a significant part of ...,https://openalex.org/W3149648688,2025-05-15 15:42:05.683112+00:00,gpt-4.1-mini,0
96,no,The text focuses on general practitioners' rol...,N/A,N/A,no,N/A,no,N/A,N/A,N/A,...,[],N/A,[],[],[],[],https://openalex.org/W2020607861,2025-05-15 15:42:05.683535+00:00,gpt-4.1-mini,0
97,no,The text is a study of emotional availability ...,N/A,N/A,no,N/A,no,N/A,N/A,N/A,...,[],N/A,[],[],[],[Emotional availability patterns vary among yo...,https://openalex.org/W2107404166,2025-05-15 15:42:05.683936+00:00,gpt-4.1-mini,0
98,no,The text discusses treatment for psychogenic n...,N/A,N/A,no,N/A,no,N/A,N/A,N/A,...,[],N/A,[],[],[],[],https://openalex.org/W2170302766,2025-05-15 15:42:05.684335+00:00,gpt-4.1-mini,0


In [10]:
output_df['is_relevant'].value_counts()

is_relevant
no     91
yes     9
Name: count, dtype: int64

In [11]:
output_df.groupby(['is_relevant', 'programme_evaluation']).size().reset_index(name='counts')

,is_relevant,programme_evaluation,counts
0,no,N/A,3
1,no,no,74
2,no,yes,14
3,yes,no,1
4,yes,yes,8


In [13]:
final_cols = ['id',
       'timestamp',
       'model',
       'temperature',
       'publication_year',
       'authorships',
       'doi',
       'title',
       'text',
       'is_retracted',
       'cited_by_count',
       'author_countries',
    'is_relevant',
              'is_relevant_reason',
              'programme_name',
              'summary',
       'programme_evaluation',
       'programme_evaluation_desc',
       'digital_component',
       'digital_component_description',
       'country',
       'age_range',
       'age_range_numerical',
       'child_outcomes_targeted',
       'parent_outcomes_targeted',
       'target_population',
       'disadvantaged_groups_engaged',
       'engagement_strategies',
       'engagement_success',
       'barriers_to_engagement',
       'evidence_of_effectiveness',
       'limitations',
       'other_learnings']

In [14]:
df_checked = (
    pd.read_json("afs_open_alex_scan_1.jsonl", lines=True)
    .merge(data_df[['id', 'publication_year', 'authorships', 'doi', 'title', 'text', 'is_retracted', 'cited_by_count']], on='id', how='left')
    .assign(
        author_countries = lambda df: df['authorships'].apply(extract_author_countries),
    )
    # convert lists to comma separated strings
    # .assign(
    #     child_outcomes = lambda df: df['child_outcomes'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
    #     child_outcomes_category = lambda df: df['child_outcomes_category'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
    #     parent_outcomes = lambda df: df['parent_outcomes'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
    #     parent_outcomes_category = lambda df: df['parent_outcomes_category'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
    #     evidence = lambda df: df['evidence'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
    #     pros = lambda df: df['pros'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
    #     cons = lambda df: df['cons'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
    #     author_countries = lambda df: df['author_countries'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
    # )
    .query("is_retracted == False")
)[final_cols]
df_checked



,id,timestamp,model,temperature,publication_year,authorships,doi,title,text,is_retracted,...,child_outcomes_targeted,parent_outcomes_targeted,target_population,disadvantaged_groups_engaged,engagement_strategies,engagement_success,barriers_to_engagement,evidence_of_effectiveness,limitations,other_learnings
0,https://openalex.org/W2163583898,2025-05-15 15:40:47.354145+00:00,gpt-4.1-mini,0,2005,"[{'author_position': 'first', 'author': {'id':...",https://doi.org/10.1177/0022427804271931,School Climate Predictors of School Disorder: ...,School Climate Predictors of School Disorder: ...,False,...,[],[],N/A,[],[],N/A,[],[],[],[]
1,https://openalex.org/W2160350695,2025-05-15 15:40:47.359249+00:00,gpt-4.1-mini,0,2010,"[{'author_position': 'first', 'author': {'id':...",https://doi.org/10.1111/j.1365-2214.2010.01067.x,Parental perspectives on caring for a child wi...,Parental perspectives on caring for a child wi...,False,...,[],[],Parents of children with chronic kidney disease,[],[],N/A,[],[],[],[Parents need better support structures to cop...
2,https://openalex.org/W2771031857,2025-05-15 15:40:47.359888+00:00,gpt-4.1-mini,0,2017,"[{'author_position': 'first', 'author': {'id':...",https://doi.org/10.1111/jmwh.12674,Integrating Trauma‐Informed Care Into Maternit...,Integrating Trauma‐Informed Care Into Maternit...,False,...,[],[],N/A,[],[],N/A,[],[],[],[]
3,https://openalex.org/W2074352831,2025-05-15 15:40:47.360462+00:00,gpt-4.1-mini,0,2000,"[{'author_position': 'first', 'author': {'id':...",https://doi.org/10.1037/0893-3200.14.1.5,Parental factors and the young child's postdiv...,Parental factors and the young child's postdiv...,False,...,[],[],N/A,[],[],N/A,[],[],[],[]
4,https://openalex.org/W2150034613,2025-05-15 15:40:47.360953+00:00,gpt-4.1-mini,0,2002,"[{'author_position': 'first', 'author': {'id':...",https://doi.org/10.1136/jech.56.7.529,Genetics and social class,Genetics and social class <b>Objective:</b> To...,False,...,[],[],N/A,[],[],N/A,[],[],[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,https://openalex.org/W3149648688,2025-05-15 15:42:05.683112+00:00,gpt-4.1-mini,0,2013,"[{'author_position': 'first', 'author': {'id':...",NaN,Improving Skills Development in the Informal S...,Improving Skills Development in the Informal S...,False,...,[],[],Workers in the informal sector in Sub-Saharan ...,[],[],N/A,[],[],[],[The informal sector is a significant part of ...
96,https://openalex.org/W2020607861,2025-05-15 15:42:05.683535+00:00,gpt-4.1-mini,0,2008,"[{'author_position': 'first', 'author': {'id':...",https://doi.org/10.1080/02813430802588907,What is the role of the general practitioner t...,What is the role of the general practitioner t...,False,...,[],[],N/A,[],[],N/A,[],[],[],[]
97,https://openalex.org/W2107404166,2025-05-15 15:42:05.683936+00:00,gpt-4.1-mini,0,2005,"[{'author_position': 'first', 'author': {'id':...",https://doi.org/10.1002/imhj.20057,Patterns of emotional availability among young...,Patterns of emotional availability among young...,False,...,[],[],Young mothers under 21 years at child's birth,[Young mothers under 21 years],[],N/A,[],[],[],[Emotional availability patterns vary among yo...
98,https://openalex.org/W2170302766,2025-05-15 15:42:05.684335+00:00,gpt-4.1-mini,0,2013,"[{'author_position': 'first', 'author': {'id':...",https://doi.org/10.1111/epi.12106,Management of psychogenic nonepileptic seizures,Management of psychogenic nonepileptic seizure...,False,...,[],[],N/A,[],[],N/A,[],[],[],[]


In [15]:
df_checked.to_csv('test_output.csv', index=False)